In [6]:
%pip install pandas numpy
import pandas as pd
import numpy as np

gain_df = pd.read_csv(r"c:\Users\Mrunal\UST_Analog_automation-main\models\Junaid\opam Transformation 5\gain_clean.csv")
ugf_df  = pd.read_csv(r"c:\Users\Mrunal\UST_Analog_automation-main\models\Junaid\opam Transformation 5\ugf_clean.csv")
pm_df   = pd.read_csv(r"c:\Users\Mrunal\UST_Analog_automation-main\models\Junaid\opam Transformation 5\pm_clean.csv")

X = gain_df[["a", "b", "c", "d"]].values

Y = np.column_stack([
    gain_df["gain"].values,
    ugf_df["ugf"].values,
    pm_df["pm"].values
])

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## METHOD 1: Multi-Output XGBoost (Boosting + Tree Ensemble)

In [8]:
%pip install xgboost scikit-learn joblib

from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

base_model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model = MultiOutputRegressor(base_model)
model.fit(X_train, Y_train)

preds = model.predict(X_test)

print("R2 Gain:", r2_score(Y_test[:,0], preds[:,0]))
print("R2 UGF :", r2_score(Y_test[:,1], preds[:,1]))
print("R2 PM  :", r2_score(Y_test[:,2], preds[:,2]))

joblib.dump(model, "xgb_multioutput.pkl")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.4/72.0 MB 11.8 MB/s eta 0:00:06
   -- ------------------------------------- 4.7/72.0 MB 11.8 MB/s eta 0:00:06
   --- ------------------------------------ 7.1/72.0 MB 11.4 MB/s eta 0:00:06
   ----- ---------------------------------- 9.7/72.0 MB 11.5 MB/s eta 0:00:06
   ------ --------------------------------- 12.1/72.0 MB 11.5 MB/s eta 0:00:06
   -------- ------------------------------- 14.4/72.0 MB 11.5 MB/s eta 0:00:06
   --------- ------------------------------ 17.0/72.0 MB 11.4 MB/s eta 0:00:05
   ---------- ----------------------------- 19.4/72.0 MB 11.4 MB/s eta 0:00:05
   ------------ --------------------------- 21.8/72.0 MB 11.4 MB/s eta 0:00:05
   ------------- -------------------------- 24.1/72.0 MB 11.3 MB/s eta 0:00:05
   -------------- ------------------------- 26.7/72.0 MB 11.4 MB/s eta 0:00:04
   ---------------- ----------------------- 29.1/72.0 MB 11.4 MB/

['xgb_multioutput.pkl']

## METHOD 2: Multi-Task Neural Network + AdamW Optimizer

In [1]:
%pip install tensorflow-cpu

import numpy as np
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# -------------------------
# Scaling
# -------------------------
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X)
Y_scaled = y_scaler.fit_transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y_scaled, test_size=0.2, random_state=42
)

# -------------------------
# Model
# -------------------------
inputs = Input(shape=(4,))

x = Dense(64, activation="relu")(inputs)
x = Dense(64, activation="relu")(x)
x = Dense(32, activation="relu")(x)

outputs = Dense(3, activation="linear")(x)

model = Model(inputs=inputs, outputs=outputs)

# -------------------------
# Compile
# -------------------------
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),
    loss="mse"
)

model.summary()

# -------------------------
# Train
# -------------------------
model.fit(
    X_train,
    Y_train,
    validation_split=0.1,
    epochs=200,
    batch_size=32,
    verbose=1
)

# -------------------------
# Save
# -------------------------
model.save("multitask_nn_gain_ugf_pm.h5")

ERROR: Could not find a version that satisfies the requirement tensorflow-cpu (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow-cpu


Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'tensorflow'

## METHOD 3: Multi-Output SVR (Kernel + SMO Optimizer)

In [15]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", MultiOutputRegressor(
        SVR(kernel="rbf", C=100, gamma="scale")
    ))
])

pipeline.fit(X_train, Y_train)
preds = pipeline.predict(X_test)

print("R2 Gain:", r2_score(Y_test[:,0], preds[:,0]))
print("R2 UGF :", r2_score(Y_test[:,1], preds[:,1]))
print("R2 PM  :", r2_score(Y_test[:,2], preds[:,2]))

joblib.dump(pipeline, "svr_multioutput.pkl")


R2 Gain: 0.8534595580615427
R2 UGF : -0.26489353706646623
R2 PM  : 0.9694714124437908


['svr_multioutput.pkl']

## METHOD 4: Multi-Output Gaussian Process (Probabilistic)

In [2]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

kernel = ConstantKernel(1.0) * RBF(1.0)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=5
)

model = MultiOutputRegressor(gpr)
model.fit(X_train, Y_train)

preds = model.predict(X_test)

print("R2 Gain:", r2_score(Y_test[:,0], preds[:,0]))
print("R2 UGF :", r2_score(Y_test[:,1], preds[:,1]))
print("R2 PM  :", r2_score(Y_test[:,2], preds[:,2]))

joblib.dump(model, "gpr_multioutput.pkl")


NameError: name 'X' is not defined